# HERMES — complete Kaggle reproduction & thesis analysis notebook (Python 3.12 runtime)

**One notebook, arranged from setup to research export.** Run cells **in order** in a Kaggle notebook with **Internet ON** and **one NVIDIA GPU** (Tesla T4 recommended). This notebook accommodates Kaggle's **Python 3.13 kernel** by using a separate **Python 3.12** interpreter for all model inference. It does **not modify the HERMES inference algorithms** or disable rotary key corrections.

## What is included

1. Environment bootstrap & exact dependency/commit verification.
2. Clean HERMES source checkout and model acquisition using a temporary cache.
3. Recovery of the three *existing* 50-video CSVs from Kaggle inputs/backups, **before GPU use**.
4. Original 10-video six-budget ablation (optional to repeat), 50-video HERMES evaluations if originals are unavailable.
5. Paired 4K/6K/50K statistics, exact McNemar test, video-cluster bootstrap, discordance casebook.
6. Original **non-HERMES** LLaVA-OneVision-0.5B *native full-prefix* control on the same 50 videos/questions, with timestamp cutoff and explicit OOM handling.
7. Matched-key comparisons, optional full 498-video StreamingBench in resumable shards, summary figures and integrity-checked ZIP archive.

**Scientific cautions:** Native full-prefix inference is a *backbone control*, not the same online streaming inference protocol as HERMES. Differences in prompting, history and input processing must be checked before causal comparisons. HERMES KV=50000 is **not** the original LLaVA baseline and did trigger cache compression in the prior 50-video study. Previous accuracy/memory values are historical, **not fabricated into any new CSV**.

**What a notebook file cannot contain:** model weights (~GB), StreamingBench videos (~many GB), or missing raw 50-video prediction files. These are downloaded from public sources or recovered from Kaggle inputs at runtime. Kaggle sessions and GPU quotas may require multiple runs. Saving a notebook version with **outputs included** is essential between sessions.

**Question identity correction (17 October 2026):** StreamingBench may repeat identical question *text* within a video with different options and answer labels. `(video_id, question)` is NOT unique. HERMES CSVs are paired using `(video_id, qa_index)` where `qa_index` is the zero-based within-video row position; all three runs are checked for matching question, option list, and reference answer. Once the source annotation is loaded, each CSV row is independently checked against the question at that ordinal. Native LLaVA uses the same ordinal for checkpointing. This avoids rerunning valid results simply due to generic repeated prompts.


## 1. Master configuration

For the quickest continuation, attach your previously saved Kaggle output ZIP or the three CSVs under **Add Input**. If the original 50-video files cannot be recovered, `REBUILD_MISSING_50=True` runs the missing experiments rather than inventing data. Change optional expensive-stage switches only when ready.

In [14]:
from pathlib import Path
import os, sys, subprocess, json, re, zipfile, hashlib, shutil, time, gc, math, platform, copy
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.stats import binomtest
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_ID='llava-hf/llava-onevision-qwen2-0.5b-ov-hf'
HERMES_URL='https://github.com/haowei-freesky/HERMES.git'
HERMES_SHA='8d699b16a6bedb9086c1b39ec4253c6a1d1ce789'
TF_SHA='66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
DATASET_ID='mjuicem/StreamingBench'
FPS=0.5
SEED=2024
KV_50=(4000,6000,50000)
KV_10=(500,1000,2000,4000,6000,50000)
FULL_BUDGETS=(4000,6000)

# Control substantial GPU use.
REBUILD_MISSING_50=True     # Auto-run only the HERMES 50-video budgets not recovered.
RUN_PILOT_10=False          # Existing historical six-budget study normally needn't rerun.
RUN_NATIVE_BASELINE_50=True # Native full-prefix LLaVA; stop honestly if GPU memory is insufficient.
RUN_FULL_HERMES=False       # Turn on for shard-wise ~498-video experiments after 50-video validation.
MAX_NEW_SHARDS_THIS_SESSION=1  # Limits full-benchmark work per Kaggle session.
RETAIN_EXTRACTED_SHARDS=False
BOOTSTRAP_REPS=10000

WORK=Path('/kaggle/working'); TEMP=Path('/kaggle/temp'); INPUT=Path('/kaggle/input')
VENV=TEMP/'hermes_py312'; PY312=VENV/'bin/python'
ROOT=WORK/'HERMES_MASTER'
RESULTS=ROOT/'results'; DATA=ROOT/'data'; LOGS=WORK/'HERMES_MASTER_LOGS'
EXPORTS=WORK/'HERMES_MASTER_EXPORTS'; FIGURES=EXPORTS/'figures'
ARCHIVE_TEMP=TEMP/'streamingbench_archives'; EXTRACT_TEMP=TEMP/'streamingbench_extracted'
HF_HOME=TEMP/'hermes_hf_home'; MODEL_CACHE=TEMP/'hermes_hf_model_cache'
for p in [WORK,TEMP,EXPORTS,FIGURES,ARCHIVE_TEMP,EXTRACT_TEMP,LOGS]:p.mkdir(parents=True,exist_ok=True)

def run(args,*,cwd=None,env=None,check=True):
    p=subprocess.run([str(a) for a in args],cwd=cwd,env=env,text=True,capture_output=True)
    if check and p.returncode:
        raise RuntimeError('COMMAND FAILED: '+repr(args)+'\n'+(p.stdout+p.stderr)[-4500:])
    return p

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

print('Kernel:',sys.version.split()[0],sys.executable)
print('Python 3.12:',run(['/usr/bin/python3.12','--version'],check=False).stdout.strip())
print('Work space:',WORK,'Disk free GiB:',round(shutil.disk_usage(WORK).free/1024**3,2))
print('GPU:'); print(run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],check=False).stdout)
print('Rebuild missing 50:',REBUILD_MISSING_50,'Full:',RUN_FULL_HERMES)


Kernel: 3.13.15 /usr/bin/python3
Python 3.12: Python 3.12.3
Work space: /kaggle/working Disk free GiB: 19.49
GPU:
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

Rebuild missing 50: True Full: False


## 2. Recover existing HERMES 50-video results (CPU only)

Looks for `streamingbench_50_kv4000/1_0.csv`, `...kv6000/...`, and `...kv50000/...` in the current/legacy Kaggle working directories, attached input directories and ZIPs. **Never** treat the single 10-video `1_0.csv` as a 50-video result.

In [15]:
RECOVERY=[]
def wanted_path(kv):return RESULTS/f'streamingbench_50_kv{kv}'/'1_0.csv'
def valid50(p):
    try:
        d=pd.read_csv(p)
        if not {'video_id','question','qa_acc'}.issubset(d.columns):return False
        if len(d)!=250 or d.video_id.nunique()!=50:return False
        # Repeated question text is common within a video. Row position identifies QA.
        if d[['video_id','question','qa_acc']].isna().any().any():return False
        if not d.groupby('video_id').size().eq(5).all():return False
        return pd.to_numeric(d.qa_acc,errors='coerce').isin([0,100]).all()
    except Exception:return False

# Locate all files once (large inputs may contain numerous files).
roots=[WORK/'HERMES', WORK/'HERMES_FINAL', WORK/'HERMES_repro', INPUT]
files=[]; zips=[]
for root in roots:
    if root.exists():
        files.extend(root.rglob('1_0.csv'))
        # Large video ZIPs are not unpacked; only search backups likely to hold results.
        zips.extend([p for p in root.rglob('*.zip') if 'streamingbench' not in p.name.lower()])

for kv in KV_50:
    dest=wanted_path(kv)
    if valid50(dest):
        RECOVERY.append({'kv':kv,'source':'existing working','path':str(dest)}); continue
    suffix=f'streamingbench_50_kv{kv}/1_0.csv'
    candidates=[p for p in files if p.as_posix().endswith(suffix) and valid50(p)]
    zip_candidates=[]
    if not candidates:
        for zpath in zips:
            try:
                with zipfile.ZipFile(zpath) as z:
                    names=[n for n in z.namelist() if n.replace('\\','/').endswith(suffix)]
                    if len(names)>1:raise ValueError(f'ambiguous matching entries in {zpath}')
                    for name in names:zip_candidates.append((zpath,name))
            except zipfile.BadZipFile:continue
    if len(candidates)+len(zip_candidates)>1:
        print('WARNING: multiple candidates for KV',kv,'; refusing ambiguous recovery.')
    elif len(candidates)==1:
        dest.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(candidates[0],dest)
        RECOVERY.append({'kv':kv,'source':str(candidates[0]),'path':str(dest)})
    elif len(zip_candidates)==1:
        zpath,name=zip_candidates[0]
        dest.parent.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zpath) as z, z.open(name) as source, dest.open('wb') as target:
            shutil.copyfileobj(source,target)
        if not valid50(dest):
            dest.unlink();raise ValueError(f'Invalid recovered CSV from {zpath}: {name}')
        RECOVERY.append({'kv':kv,'source':str(zpath)+'::'+name,'path':str(dest)})

missing_50=[kv for kv in KV_50 if not valid50(wanted_path(kv))]
print('Recovered or present:',[x['kv'] for x in RECOVERY]);print('Missing:',missing_50)
pd.DataFrame(RECOVERY).to_csv(EXPORTS/'recovery_manifest.csv',index=False)
if missing_50 and not REBUILD_MISSING_50:
    raise FileNotFoundError('Missing raw 50-video CSVs. Attach previous Kaggle outputs or enable REBUILD_MISSING_50 in configuration.')


Recovered or present: [4000]
Missing: [6000, 50000]


## 3. Analysis utilities (CPU)

Video-grouped bootstrap intervals account for correlated questions within the same video. The exact discordance/binomial test is an **exploratory sensitivity test**, not a substitute for the video-cluster interval.
**Note:** `qa_index` (within-video position) is validated against the original source annotation later in the notebook, after the 50-video shard has been prepared. Do not reorder rows in the CSV before that audit.


In [16]:
KEY=['video_id','qa_index']
def read_hermes_csv(p, expected_videos=None, expected_rows=None):
    d=pd.read_csv(p).copy()
    assert {'video_id','question','qa_acc'}.issubset(d.columns),list(d.columns)
    d['video_id']=d['video_id'].astype(str);d['question']=d['question'].astype(str).str.strip()
    # HERMES writes each video's questions in annotation order. A generic question
    # text can occur multiple times with different choices and answer keys.
    d['qa_index']=d.groupby('video_id', sort=False).cumcount().astype(int)
    assert not d.duplicated(KEY).any(),f'Duplicated positional keys in {p}'
    assert d['question'].notna().all()
    # If source annotation was already loaded, validate each ordinal against it.
    if 'ANNOTATION_LOOKUP' in globals():
        validate_annotation_positions(d, ANNOTATION_LOOKUP, label=str(p))
    v=pd.to_numeric(d.qa_acc,errors='coerce')
    assert v.isin([0,100]).all(),f'Invalid HERMES qa_acc in {p}'
    d['correct']=v.eq(100).astype(int)
    if expected_videos is not None:assert d.video_id.nunique()==expected_videos,(p,d.video_id.nunique())
    if expected_rows is not None:assert len(d)==expected_rows,(p,len(d))
    # Audit direct parsed choices where available; do not silently override official evaluator.
    if {'pred_choice','correct_choice'}.issubset(d):
        ok=(d.pred_choice.astype(str).str.strip().str.upper()==d.correct_choice.astype(str).str.strip().str.upper())
        mismatch=int((ok!=d.correct.astype(bool)).sum())
        print(Path(p).parent.name,': parsed-choice vs qa_acc mismatches:',mismatch)
    return d

def paired(a,b,name_a,name_b,reps=BOOTSTRAP_REPS,seed=SEED):
    assert set(map(tuple,a[KEY].values))==set(map(tuple,b[KEY].values)),'Non-matching question keys.'
    # Guard against silently pairing different questions at the same ordinal.
    aa=a.set_index(KEY).sort_index();bb=b.set_index(KEY).sort_index()
    for field in ('question','choices','correct_choice','answer','task'):
        if field in aa and field in bb:
            left=aa[field].fillna('').astype(str).str.strip()
            right=bb[field].fillna('').astype(str).str.strip()
            bad=left.ne(right)
            assert not bad.any(),f'Question/metadata mismatch in {field}: {bad[bad].index.tolist()[:5]}'
    x=a[KEY+['correct']].rename(columns={'correct':'a'}).merge(
      b[KEY+['correct']].rename(columns={'correct':'b'}),on=KEY,validate='one_to_one')
    n11=int(((x.a==1)&(x.b==1)).sum());n00=int(((x.a==0)&(x.b==0)).sum())
    n10=int(((x.a==1)&(x.b==0)).sum());n01=int(((x.a==0)&(x.b==1)).sum())
    exact=binomtest(n10,n10+n01,.5).pvalue if (n10+n01) else 1.
    vs=x.assign(delta=x.a-x.b).groupby('video_id').delta.agg(['sum','count'])
    rng=np.random.default_rng(seed)
    sums=vs['sum'].to_numpy(float);counts=vs['count'].to_numpy(float)
    # Bounded memory: 10000 x 498 indices is comfortably small.
    draws=rng.integers(0,len(sums),size=(reps,len(sums)))
    boot=100*sums[draws].sum(axis=1)/counts[draws].sum(axis=1)
    res={'A':name_a,'B':name_b,'n':len(x),'n_videos':x.video_id.nunique(),
         'both_correct':n11,'both_wrong':n00,'A_only_correct':n10,'B_only_correct':n01,
         'accuracy_A_pct':100*x.a.mean(),'accuracy_B_pct':100*x.b.mean(),
         'difference_A_minus_B_pp':100*(x.a-x.b).mean(),
         'video_cluster_CI95_low_pp':float(np.quantile(boot,.025)),
         'video_cluster_CI95_high_pp':float(np.quantile(boot,.975)),
         'exact_discordance_p_exploratory':exact}
    x['comparison']=f'{name_a}_vs_{name_b}'
    x['case_type']=np.select([x.a.eq(1)&x.b.eq(1),x.a.eq(0)&x.b.eq(0),x.a.eq(1)&x.b.eq(0),x.a.eq(0)&x.b.eq(1)],
                             ['both_correct','both_wrong','A_only_correct','B_only_correct'],default='invalid')
    return res,x

def analyze50():
    dfs={k:read_hermes_csv(wanted_path(k),50,250) for k in KV_50}
    sets=[set(map(tuple,d[KEY].values)) for d in dfs.values()]
    assert sets[0]==sets[1]==sets[2],'50-video runs do not contain exactly matching questions.'
    rows=[];cases=[]
    for a,b in [(4000,6000),(4000,50000),(6000,50000)]:
        r,c=paired(dfs[a],dfs[b],f'HERMES_{a}',f'HERMES_{b}')
        rows.append(r);cases.append(c)
    comparison=pd.DataFrame(rows); casebook=pd.concat(cases,ignore_index=True)
    meta_cols=[c for c in ['video_id','qa_index','question','answer','choices','task','pred_answer','pred_choice','correct_choice'] if c in dfs[4000].columns]
    casebook=casebook.merge(dfs[4000][meta_cols],on=KEY,how='left',validate='many_to_one')
    pd.DataFrame([{ 'method':f'HERMES KV={k}', 'n':len(d), 'accuracy_pct':100*d.correct.mean()} for k,d in dfs.items()]).to_csv(EXPORTS/'hermes_50_accuracy.csv',index=False)
    comparison.to_csv(EXPORTS/'paired_50_tests.csv',index=False)
    casebook.to_csv(EXPORTS/'paired_50_casebook.csv',index=False)
    all_three=dfs[4000][KEY+['question','correct']].rename(columns={'correct':'ok4'})
    for k,col in [(6000,'ok6'),(50000,'ok50')]:
        all_three=all_three.merge(dfs[k][KEY+['correct']].rename(columns={'correct':col}),on=KEY,validate='one_to_one')
    all_three['pattern']=all_three[['ok4','ok6','ok50']].astype(str).agg(''.join,axis=1)
    all_three.to_csv(EXPORTS/'all_three_50_correctness.csv',index=False)
    print('Three-budget correctness patterns (4K,6K,50K):');display(all_three.pattern.value_counts().rename_axis('pattern').to_frame('questions'))
    display(comparison.round(4))
    return dfs,comparison,casebook

# Compute the zero-GPU result immediately when recovered; otherwise it follows the missing runs.
if not missing_50:
    h50, paired50, casebook50 = analyze50()
    print('CPU paired analysis completed without model loading.')
else:
    h50=paired50=casebook50=None
    print('Paired CPU analysis deferred until missing CSVs are produced.')


Paired CPU analysis deferred until missing CSVs are produced.


## 4. Establish isolated Python 3.12 / GPU dependencies

Do **not** install the old Transformers into Python 3.13. All Torch/Transformers/model operations below run in subprocesses through `/kaggle/temp/hermes_py312/bin/python`. Python 3.12 venv creation uses `uv` so `ensurepip` is unnecessary. A full CUDA PyTorch wheel may take several GB of temporary disk.

In [17]:
# Create/reuse the isolated interpreter. Kaggle's Python 3.13 kernel is unchanged.
if not (Path('/usr/bin/python3.12')).exists():
    raise RuntimeError('Python 3.12 is not available in this Kaggle image.')
try:run([sys.executable,'-m','uv','--version'])
except RuntimeError:
    run([sys.executable,'-m','pip','install','-q','uv'])
UV=[sys.executable,'-m','uv']
if not PY312.exists():run(UV+['venv',str(VENV),'--python','/usr/bin/python3.12','--seed'])
assert '3.12.' in run([PY312,'--version']).stdout
INSTALL_ENV=os.environ.copy()
INSTALL_ENV.update({'UV_CACHE_DIR':str(TEMP/'hermes_uv_cache'),'UV_LINK_MODE':'copy','TMPDIR':str(TEMP)})

def uv_install(packages, extra_args=()):
    command=UV+['pip','install','--python',str(PY312)]+list(extra_args)+list(packages)
    p=run(command,env=INSTALL_ENV)
    print('Installed/verified:',', '.join(packages))
    if p.stderr:print(p.stderr[-1400:])

# Separate index call is necessary for the CUDA 12.8 wheel.
uv_install(['torch==2.10.0+cu128'],['--index-url','https://download.pytorch.org/whl/cu128'])
uv_install(['numpy==1.26.4','tokenizers==0.19.1','huggingface-hub==0.34.4',
            'accelerate==0.34.2','decord','logzero','pillow','sentencepiece','safetensors',
            'packaging','filelock','regex','requests','pyyaml','tqdm','fsspec','protobuf',
            'einops','psutil','pandas','scipy','av'])
uv_install(['git+https://github.com/huggingface/transformers.git@'+TF_SHA],['--no-deps'])
print('Python 3.12 setup finished')


Installed/verified: torch==2.10.0+cu128
Using Python 3.12.3 environment at: /kaggle/temp/hermes_py312
Checked 1 package in 69ms

Installed/verified: numpy==1.26.4, tokenizers==0.19.1, huggingface-hub==0.34.4, accelerate==0.34.2, decord, logzero, pillow, sentencepiece, safetensors, packaging, filelock, regex, requests, pyyaml, tqdm, fsspec, protobuf, einops, psutil, pandas, scipy, av
Using Python 3.12.3 environment at: /kaggle/temp/hermes_py312
Resolved 55 packages in 102ms
Uninstalled 1 package in 5ms
Installed 1 package in 29ms
 ~ decord==0.6.0

Installed/verified: git+https://github.com/huggingface/transformers.git@66bc4def9505fa7c7fe4aa7a248c34a026bb552b
Using Python 3.12.3 environment at: /kaggle/temp/hermes_py312
Checked 1 package in 1ms

Python 3.12 setup finished


In [18]:
# Verify the interpreter which actually executes HERMES (not Kaggle's kernel).
check_code=r"""
import sys,torch,transformers,tokenizers,huggingface_hub,importlib.metadata as md,json
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
import inspect
assert sys.version_info[:2]==(3,12),sys.version
assert transformers.__version__=='4.45.0.dev0',transformers.__version__
assert tokenizers.__version__=='0.19.1',tokenizers.__version__
assert huggingface_hub.__version__=='0.34.4',huggingface_hub.__version__
assert torch.cuda.is_available(),'PyTorch 3.12 cannot see CUDA'
meta=md.distribution('transformers').read_text('direct_url.json')
assert meta and json.loads(meta)['vcs_info']['commit_id']=='66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
assert 'seq_len' in str(inspect.signature(Qwen2RotaryEmbedding.forward))
print('Interpreter:',sys.executable,'Torch:',torch.__version__)
print('GPU:',torch.cuda.get_device_name(0))
print('Transformers:',transformers.__version__,'source:',meta)
"""
print(run([PY312,'-c',check_code],env={**os.environ,'CUDA_VISIBLE_DEVICES':'0','HF_HOME':str(HF_HOME)}).stdout)


Interpreter: /kaggle/temp/hermes_py312/bin/python Torch: 2.10.0+cu128
GPU: Tesla T4
Transformers: 4.45.0.dev0 source: {"url":"https://github.com/huggingface/transformers.git","vcs_info":{"vcs":"git","commit_id":"66bc4def9505fa7c7fe4aa7a248c34a026bb552b","requested_revision":"66bc4def9505fa7c7fe4aa7a248c34a026bb552b"}}



## 5. Obtain clean official HERMES source

Checks out the exact recorded HERMES commit and audits changes. Only a model-directory **symlink** is added (not a functional edit). Source files such as `reindex_1d.py`, `llavaov_hermes.py`, and `video_qa/base.py` are **never patched**.

In [20]:
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/HERMES_MASTER")

result = subprocess.run(
    ["git", "diff", "--name-only", "HEAD", "--"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True
)

changed = [x for x in result.stdout.splitlines() if x.strip()]

print("=== CHANGED TRACKED FILES ===")
for path in changed:
    print(path)

print("\n=== CHANGE CLASSIFICATION ===")

python_sources = [p for p in changed if p.endswith(".py")]
compiled_files = [p for p in changed if p.endswith(".pyc")]
other_files = [
    p for p in changed
    if not p.endswith((".py", ".pyc"))
]

print("Python source files:", python_sources)
print("Compiled Python files:", compiled_files)
print("Other tracked files:", other_files)

if python_sources:
    print("\n=== PYTHON SOURCE DIFF ===")
    diff = subprocess.run(
        ["git", "diff", "HEAD", "--", *python_sources],
        cwd=repo,
        capture_output=True,
        text=True
    )
    print(diff.stdout[:6000])

=== CHANGED TRACKED FILES ===
inference/__pycache__/abstract_hermes.cpython-312.pyc
inference/__pycache__/llavaov_hermes.cpython-312.pyc
video_qa/__pycache__/base.cpython-312.pyc

=== CHANGE CLASSIFICATION ===
Python source files: []
Compiled Python files: ['inference/__pycache__/abstract_hermes.cpython-312.pyc', 'inference/__pycache__/llavaov_hermes.cpython-312.pyc', 'video_qa/__pycache__/base.cpython-312.pyc']
Other tracked files: []


In [21]:
if not (ROOT/'.git').exists():
    # CSV recovery may have already created ROOT/results. Clone into staging,
    # then merge source files while keeping the recovered predictions intact.
    if ROOT.exists() and any(p.name not in {'results','data'} for p in ROOT.iterdir()):
        raise RuntimeError(f'{ROOT} has unrecognized files without Git metadata: inspect before continuing.')
    staging=TEMP/'hermes_master_source_staging'
    if staging.exists():shutil.rmtree(staging)
    run(['git','clone',HERMES_URL,str(staging)])
    shutil.copytree(staging,ROOT,dirs_exist_ok=True,symlinks=True)
    shutil.rmtree(staging)
now=run(['git','rev-parse','HEAD'],cwd=ROOT).stdout.strip()
if now!=HERMES_SHA:
    run(['git','checkout','--detach',HERMES_SHA],cwd=ROOT)
    now=run(['git','rev-parse','HEAD'],cwd=ROOT).stdout.strip()
assert now==HERMES_SHA,(now,HERMES_SHA)
# Verify tracked files, allowing Python-generated bytecode changes.
changed_files = run(
    ['git', 'diff', '--name-only', 'HEAD', '--'],
    cwd=ROOT
).stdout.splitlines()

unexpected_changes = [
    p for p in changed_files
    if not (p.endswith('.pyc') and '__pycache__/' in p)
]

assert not unexpected_changes, (
    f"Unexpected tracked file modifications: {unexpected_changes}"
)

print("PASS: HERMES tracked source is unchanged.")
print("Ignored generated bytecode files:", len(changed_files))
for rel in ['inference/llavaov_hermes.py','inference/reindex_1d.py','video_qa/base.py']:
    print(rel,sha256(ROOT/rel))
assert 'return key_states' not in (ROOT/'inference/reindex_1d.py').read_text().split('def apply_rotary_delta_to_keys_only')[-1][:230], 'Unexpected disabled RoPE delta patch'
print('HERMES SHA:',now,'Source fidelity check: PASS')


PASS: HERMES tracked source is unchanged.
Ignored generated bytecode files: 3
inference/llavaov_hermes.py 45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0
inference/reindex_1d.py 22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0
video_qa/base.py 795cf98e8d1125ea934dc2c9744ba1bbd3b29701f17aae2d935f382c7343b2d5
HERMES SHA: 8d699b16a6bedb9086c1b39ec4253c6a1d1ce789 Source fidelity check: PASS


### 5B. Recover native/full prediction checkpoints from saved Kaggle inputs

Restores optional prior `native_llava_50/predictions.csv` and complete or partial `full_<shard>_kv<budget>/1_0.csv` data from attached folders/ZIPs. No inference is performed. The restored files are validated when the corresponding analysis stage runs.

In [22]:
# Only files in a research output directory or research ZIP are candidates.
def recover_saved_relative(suffix,dest):
    if dest.exists() and dest.stat().st_size:return
    possible=[p for p in INPUT.rglob(suffix.rsplit('/',1)[-1]) if p.as_posix().endswith(suffix)] if INPUT.exists() else []
    if len(possible)==1:
        dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(possible[0],dest)
        print('Restored:',suffix,'from',possible[0]);return
    if len(possible)>1:raise RuntimeError('Multiple checkpoint candidates: '+suffix)
    found=[]
    for zpath in zips:
        try:
            with zipfile.ZipFile(zpath) as z:
                for name in z.namelist():
                    if name.endswith(suffix):found.append((zpath,name))
        except zipfile.BadZipFile:continue
    if len(found)==1:
        zpath,name=found[0]
        dest.parent.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zpath) as z,z.open(name) as src,dest.open('wb') as target:
            shutil.copyfileobj(src,target)
        print('Restored:',suffix,'from backup ZIP')
    elif len(found)>1:
        raise RuntimeError('Ambiguous checkpoint ZIP entries: '+suffix)

recover_saved_relative('native_llava_50/predictions.csv',RESULTS/'native_llava_50'/'predictions.csv')
for a,b in [(1,50),(51,100),(101,150),(151,200),(201,250),(251,300),(301,350),(351,400),(401,450),(451,500)]:
    for kv in FULL_BUDGETS:
        name=f'full_{a}-{b}_kv{kv}/1_0.csv'
        recover_saved_relative(name,RESULTS/name)


## 6. Download the model to temporary storage

The isolated Python 3.12 runtime downloads the regular checkpoint assets only. **ONNX exports and model copies under `/kaggle/working` are excluded**. The original HERMES model path is a symlink to the downloaded snapshot.

In [23]:
MODEL_DOWNLOADER=EXPORTS/'download_model_py312.py'
MODEL_DOWNLOADER.write_text(r"""
import json,sys
from huggingface_hub import snapshot_download
repo=sys.argv[1]; cache=sys.argv[2]
p=snapshot_download(repo_id=repo,cache_dir=cache,
  allow_patterns=['*.json','*.safetensors','*.txt','*.model','*.vocab','tokenizer.*','merges.txt'],
  ignore_patterns=['onnx/*','*.onnx','*.bin','*.msgpack'])
print('MODEL_SNAPSHOT_JSON='+json.dumps(str(p)))
""")
model_env={**os.environ,'HF_HOME':str(HF_HOME),'HF_HUB_DISABLE_XET':'1'}
p=run([PY312,str(MODEL_DOWNLOADER),MODEL_ID,str(MODEL_CACHE)],env=model_env)
lines=[s for s in p.stdout.splitlines() if s.startswith('MODEL_SNAPSHOT_JSON=')]
assert lines,'Model snapshot was not reported: '+p.stdout[-1000:]
MODEL_SNAPSHOT=Path(json.loads(lines[-1].split('=',1)[1]))
assert MODEL_SNAPSHOT.exists() and list(MODEL_SNAPSHOT.glob('*.safetensors')),'Checkpoint weights missing.'
link=ROOT/'models'/'llava-onevision-qwen2-0.5b-ov-hf'
link.parent.mkdir(parents=True,exist_ok=True)
if link.is_symlink() and link.resolve()!=MODEL_SNAPSHOT.resolve():link.unlink()
if not link.exists():link.symlink_to(MODEL_SNAPSHOT,target_is_directory=True)
assert (link/'config.json').exists()
MODEL_DIR=link
print('Model linked:',link,'->',MODEL_SNAPSHOT)
print('Snapshot identity:',MODEL_SNAPSHOT.name)


Model linked: /kaggle/working/HERMES_MASTER/models/llava-onevision-qwen2-0.5b-ov-hf -> /kaggle/temp/hermes_hf_model_cache/models--llava-hf--llava-onevision-qwen2-0.5b-ov-hf/snapshots/74dd0bf867a4cda7950c17663794267c60cf4b40
Snapshot identity: 74dd0bf867a4cda7950c17663794267c60cf4b40


## 7. Prepare 201–250 StreamingBench shard (50 videos, 250 questions)

Source: the public `mjuicem/StreamingBench` archive `Real-Time Visual Understanding_201-250.zip`; the official HERMES annotation JSON determines the *exact* 50-video question set. Downloads/extracts to `/kaggle/temp`, not the limited Kaggle output directory. **This step requires several GB of temporary space.**

In [24]:
from pathlib import PurePosixPath
ANNOTATION_MASTER=ROOT/'data'/'streamingbench'/'streamingbench_realtime.json'
assert ANNOTATION_MASTER.exists(),ANNOTATION_MASTER
annotations=json.loads(ANNOTATION_MASTER.read_text())
assert isinstance(annotations,list)
print('Annotation records:',len(annotations),'questions:',sum(len(x['conversations']) for x in annotations))

SHARD_50='Real-Time Visual Understanding_201-250.zip'

def safe_extract(z,output_dir):
    output_dir=Path(output_dir);base=output_dir.resolve()
    for item in z.infolist():
        target=(output_dir/item.filename).resolve()
        if base!=target and base not in target.parents:raise ValueError('ZIP path traversal: '+item.filename)
    z.extractall(output_dir)

def fetch_extract(archive_name):
    name=re.sub(r'[^a-zA-Z0-9_-]','_',archive_name.removesuffix('.zip'))
    dest=EXTRACT_TEMP/name
    if any(dest.rglob('*.mp4')):return dest
    dest.mkdir(parents=True,exist_ok=True)
    # Use Python 3.12 for Hugging Face Hub to avoid kernel dependency conflicts.
    dl_worker=EXPORTS/'hf_shard_dl_py312.py'
    dl_worker.write_text(r"""
import sys
from huggingface_hub import hf_hub_download
p=hf_hub_download(repo_id=sys.argv[1],repo_type='dataset',filename=sys.argv[2],local_dir=sys.argv[3])
print('ARCHIVE_JSON='+__import__('json').dumps(str(p)))
""")
    p=run([PY312,str(dl_worker),DATASET_ID,archive_name,str(ARCHIVE_TEMP)],env=model_env)
    lines=[s for s in p.stdout.splitlines() if s.startswith('ARCHIVE_JSON=')]
    if not lines:raise RuntimeError('Archive path not returned: '+p.stdout[-2000:])
    zpath=Path(json.loads(lines[-1].split('=',1)[1]))
    assert zipfile.is_zipfile(zpath),zpath
    with zipfile.ZipFile(zpath) as z:safe_extract(z,dest)
    # Delete the ZIP only after an extract that actually contains videos.
    assert any(dest.rglob('*.mp4')),'Archive contains no supported videos'
    zpath.unlink(missing_ok=True)
    return dest

def annotation_for_directory(path):
    candidates={}
    for p in Path(path).rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.mp4','.avi','.mov','.mkv'} and not p.name.startswith('._'):
            matches=re.findall(r'sample_(\d+)',str(p))
            if matches:
                sid=int(matches[-1])
                if sid in candidates:raise RuntimeError(f'Multiple video candidates for sample_{sid}')
                candidates[sid]=p
    mapped=[]
    for item in annotations:
        m=re.search(r'sample_(\d+)_real\.mp4',str(item['video_path']))
        if m and int(m.group(1)) in candidates:
            item2=copy.deepcopy(item)
            item2['video_path']=str(candidates[int(m.group(1))])
            mapped.append((int(m.group(1)),item2))
    return [r for _,r in sorted(mapped,key=lambda x:x[0])]

shard50_dir=fetch_extract(SHARD_50)
matched50=annotation_for_directory(shard50_dir)
assert len(matched50)==50,(len(matched50),'expected exactly 50 videos')
assert sum(len(x['conversations']) for x in matched50)==250
assert all(Path(x['video_path']).exists() for x in matched50)
ANNOTATION_50=DATA/'streamingbench_50_kaggle.json'
ANNOTATION_50.write_text(json.dumps(matched50,indent=2))
ANNOTATION_10=DATA/'streamingbench_10_kaggle.json'
ANNOTATION_10.write_text(json.dumps(matched50[:10],indent=2))
print('Prepared 50:',ANNOTATION_50,'first video_id:',matched50[0]['video_id'])


# Verify that positional question IDs really correspond to the source annotations.
# This is a hard gate to prevent invalid question matching or distorted statistics.
import ast
ANNOTATION_LOOKUP={str(item['video_id']):item for item in matched50}

def validate_annotation_positions(d, lookup, label='CSV'):
    tested=0
    for _,row in d.iterrows():
        vid=str(row['video_id'])
        if vid not in lookup:
            continue  # Allows 10-video and non-overlapping full benchmark shards.
        position=int(row['qa_index'])
        qa_list=lookup[vid]['conversations']
        assert 0<=position<len(qa_list),(label,vid,position,'out of bounds')
        qa=qa_list[position]
        assert str(row['question']).strip()==str(qa['question']).strip(),(
            label,vid,position,'question-order mismatch',row['question'],qa['question'])
        if 'choices' in d:
            raw=row['choices']
            csv_choices=ast.literal_eval(raw) if isinstance(raw,str) else raw
            assert isinstance(csv_choices,(tuple,list)),(label,vid,position,'bad choices')
            assert [str(c).strip() for c in csv_choices]==[str(c).strip() for c in qa['choices']],(
                label,vid,position,'choices differ')
        answers=[str(x).strip() for x in qa['choices']]
        gold=str(qa['answer']).strip()
        hits=[j for j,a in enumerate(answers) if a==gold]
        assert len(hits)==1,(label,vid,position,'ambiguous gold answer')
        if 'correct_choice' in d:
            assert str(row['correct_choice']).strip().upper()=='ABCD'[hits[0]],(
                label,vid,position,'gold-letter mismatch')
        tested+=1
    print(f'Annotation position audit: {label}: {tested} rows matched to source')

# Immediately audit any existing 50-video files, without rerunning models.
for k in KV_50:
    if wanted_path(k).exists():
        _audit_df=read_hermes_csv(wanted_path(k),50,250)
        print('Existing KV',k,'verified:',len(_audit_df),'rows')


Annotation records: 498 questions: 2495
Prepared 50: /kaggle/working/HERMES_MASTER/data/streamingbench_50_kaggle.json first video_id: 231
Annotation position audit: /kaggle/working/HERMES_MASTER/results/streamingbench_50_kv4000/1_0.csv: 250 rows matched to source
streamingbench_50_kv4000 : parsed-choice vs qa_acc mismatches: 0
Existing KV 4000 verified: 250 rows


## 8. Official HERMES inference driver and optional six-budget pilot

All HERMES runs use the **unmodified official implementation**, `--sample_fps 0.5`, `--streaming true`, one visible GPU, and exact SHA. A *valid complete* result file is reused; partial outputs are never treated as complete. Failures stop instead of being counted as successes.

In [25]:
SUBENV={**os.environ,'CUDA_VISIBLE_DEVICES':'0','HF_HOME':str(HF_HOME),
        'PYTHONPATH':str(ROOT)+os.pathsep+os.environ.get('PYTHONPATH',''),
        'TOKENIZERS_PARALLELISM':'false'}

def run_hermes(anno_path, output_name, budget, expected_videos, expected_qa):
    result_dir=RESULTS/output_name;csv=result_dir/'1_0.csv';log=LOGS/(output_name+'.log')
    if csv.exists():
        try:
            current=read_hermes_csv(csv,expected_videos,expected_qa)
            print('Validated existing:',output_name);return current
        except Exception as exc:
            raise RuntimeError(f'Incomplete or malformed existing file: {csv}; archive/rename it before rerun.') from exc
    result_dir.mkdir(parents=True,exist_ok=True)
    cmd=[PY312,'-m','video_qa.hermes_vqa','--model','llava_ov_0.5b',
        '--sample_fps',str(FPS),'--save_dir',str(result_dir),
        '--anno_path',str(anno_path),'--debug','false','--num_chunks','1',
        '--chunk_idx','0','--kv_size',str(budget),'--streaming','true']
    start=time.time()
    with log.open('w') as fh:
        proc=subprocess.run([str(x) for x in cmd],cwd=ROOT,env=SUBENV,stdout=fh,stderr=subprocess.STDOUT)
    print(output_name,'return:',proc.returncode,'elapsed minutes:',round((time.time()-start)/60,2))
    if proc.returncode:
        print(log.read_text(errors='replace')[-3500:])
        raise RuntimeError(f'HERMES command failed ({output_name}); inspect {log}')
    return read_hermes_csv(csv,expected_videos,expected_qa)

if RUN_PILOT_10:
    for kv in KV_10:
        run_hermes(ANNOTATION_10,f'streamingbench_10_kv{kv}',kv,10,50)
else:print('Historical six-budget pilot skipped; all six run configurations are available if enabled.')


Historical six-budget pilot skipped; all six run configurations are available if enabled.


## 9. Missing 50-video runs (automatic only when necessary)

If your previous three CSVs were recovered, this cell does **not** rerun the expensive 50-video work. If any are missing, it performs only those budgets, in sequence. GPU stages may require hours. Keep the notebook active; save a Kaggle version with outputs afterward.

In [26]:
for kv in KV_50:
    p=wanted_path(kv)
    if valid50(p):print('Reused HERMES KV',kv);continue
    if not REBUILD_MISSING_50:raise RuntimeError(f'Missing KV {kv} result and rebuild disabled')
    d=run_hermes(ANNOTATION_50,f'streamingbench_50_kv{kv}',kv,50,250)
    print('New HERMES accuracy KV',kv,round(100*d.correct.mean(),2),'%')
assert all(valid50(wanted_path(kv)) for kv in KV_50)
print('All 50-video source CSVs are now available and structurally valid.')


Reused HERMES KV 4000
streamingbench_50_kv6000 return: 0 elapsed minutes: 42.49
Annotation position audit: /kaggle/working/HERMES_MASTER/results/streamingbench_50_kv6000/1_0.csv: 250 rows matched to source
streamingbench_50_kv6000 : parsed-choice vs qa_acc mismatches: 0
New HERMES accuracy KV 6000 57.2 %
streamingbench_50_kv50000 return: 0 elapsed minutes: 59.75
Annotation position audit: /kaggle/working/HERMES_MASTER/results/streamingbench_50_kv50000/1_0.csv: 250 rows matched to source
streamingbench_50_kv50000 : parsed-choice vs qa_acc mismatches: 0
New HERMES accuracy KV 50000 55.2 %
All 50-video source CSVs are now available and structurally valid.


## 10. Finish 50-video paired analysis (zero additional GPU)

Creates the full paired tables, exploratory McNemar sensitivity tests, cluster-bootstrap CIs, 4K/6K/50K correctness patterns, and discordance casebook.

In [27]:
h50,paired50,casebook50=analyze50()
print('Observed data (not notebook assumptions):')
for k in KV_50:print('KV',k,'correct:',int(h50[k].correct.sum()),'out of',len(h50[k]))


Annotation position audit: /kaggle/working/HERMES_MASTER/results/streamingbench_50_kv4000/1_0.csv: 250 rows matched to source
streamingbench_50_kv4000 : parsed-choice vs qa_acc mismatches: 0
Annotation position audit: /kaggle/working/HERMES_MASTER/results/streamingbench_50_kv6000/1_0.csv: 250 rows matched to source
streamingbench_50_kv6000 : parsed-choice vs qa_acc mismatches: 0
Annotation position audit: /kaggle/working/HERMES_MASTER/results/streamingbench_50_kv50000/1_0.csv: 250 rows matched to source
streamingbench_50_kv50000 : parsed-choice vs qa_acc mismatches: 0
Three-budget correctness patterns (4K,6K,50K):


,questions
pattern,
111,133
000,101
110,8
001,4
010,2
100,1
101,1


,A,B,n,n_videos,both_correct,both_wrong,A_only_correct,B_only_correct,accuracy_A_pct,accuracy_B_pct,difference_A_minus_B_pp,video_cluster_CI95_low_pp,video_cluster_CI95_high_pp,exact_discordance_p_exploratory
0,HERMES_4000,HERMES_6000,250,50,141,105,2,2,57.2,57.2,0.0,-1.6,1.6,1.0000
1,HERMES_4000,HERMES_50000,250,50,134,103,9,4,57.2,55.2,2.0,-0.8,4.8,0.2668
2,HERMES_6000,HERMES_50000,250,50,133,102,10,5,57.2,55.2,2.0,-0.8,4.8,0.3018


Observed data (not notebook assumptions):
KV 4000 correct: 143 out of 250
KV 6000 correct: 143 out of 250
KV 50000 correct: 138 out of 250


## 11. Native LLaVA-OneVision control (standalone Python 3.12 worker)

**No HERMES code is used in this worker.** It samples the prefix at 0.5 FPS **up to the question's `end_time`**, runs the base LLaVA model directly, and evaluates a letter choice. Unlike HERMES, native prefix inference can require a large amount of GPU memory; if it fails on T4, the notebook records that failure rather than silently truncating frames and claiming equivalence. This is a *native full-prefix control*, not a faithful streaming native implementation. Prompt/frame preprocessing differences are documented limitations.

The native worker processes **one question at a time** and atomically checkpoints its CSV, enabling reruns on the same saved Kaggle outputs.

In [28]:
WORKER_SOURCE = r"""
import argparse, json, re, traceback, csv, time, os
from pathlib import Path
import numpy as np
import torch
from decord import VideoReader,cpu
from transformers import AutoProcessor,LlavaOnevisionForConditionalGeneration

parser=argparse.ArgumentParser()
parser.add_argument('--records',required=True);parser.add_argument('--output',required=True)
parser.add_argument('--model',required=True);parser.add_argument('--limit',type=int,default=0)
parser.add_argument('--fps',type=float,default=.5)
a=parser.parse_args()
assert torch.cuda.is_available(),'CUDA unavailable in native worker'
assert a.fps>0
records=json.loads(Path(a.records).read_text())
output=Path(a.output);output.parent.mkdir(parents=True,exist_ok=True)
processor=AutoProcessor.from_pretrained(a.model,trust_remote_code=True)
model=LlavaOnevisionForConditionalGeneration.from_pretrained(a.model,torch_dtype=torch.float16,low_cpu_mem_usage=True,trust_remote_code=True).to('cuda:0').eval()

def sample(path,limit):
    vr=VideoReader(path,ctx=cpu(0))
    fps=float(vr.get_avg_fps());duration=len(vr)/fps
    assert limit is not None,'No question cutoff: refusing to use future frames'
    end=min(float(limit),duration)
    assert end>=0
    t=np.arange(0,end+1.e-7,1./a.fps)
    ids=np.unique(np.clip(np.floor(t*fps).astype(int),0,len(vr)-1))
    if not len(ids):ids=np.array([0])
    arr=vr.get_batch(ids).asnumpy()
    return arr,len(ids),duration

def choice(s):
    s=str(s).strip().upper()
    patterns=[r'\b(?:OPTION|ANSWER|CHOICE)\s*[:\-]?\s*\(?([A-D])\)?\b',r'^\s*\(?([A-D])\)?(?:[.):-]|\s|$)',r'\(([A-D])\)']
    for p in patterns:
        m=re.search(p,s)
        if m:return m.group(1)
    return ''

def old_rows():
    if not output.exists():return []
    with output.open(newline='') as f:return list(csv.DictReader(f))
fields=['video_id','qa_index','question','task','end_time_s','gold_choice','prediction_text','parsed_choice','correct','sampled_frames','duration_s','elapsed_s','peak_allocated_gb','peak_reserved_gb','status']
rows=old_rows()
if rows and any('qa_index' not in x or x['qa_index']=='' for x in rows):
    raise RuntimeError('Legacy native CSV lacks qa_index: preserve it, then migrate or rename before resuming.')
done={(str(x['video_id']),int(x['qa_index'])) for x in rows if x.get('status')=='ok'}
processed=0
for rec in records:
    key=(str(rec['video_id']),int(rec['qa_index']))
    if key in done:continue
    if a.limit and processed>=a.limit:break
    processed+=1
    try:
        frames,nframes,duration=sample(rec['video_path'],rec['end_time_s'])
        letters='ABCD'
        prompt_q=str(rec['question']).strip()+'\\n'+ '\\n'.join(f'({letters[i]}) {s}' for i,s in enumerate(rec['choices'][:4]))+'\\nAnswer using only the option letter.'
        chat=[{'role':'user','content':[{'type':'video'},{'type':'text','text':prompt_q}]}]
        prompt=processor.apply_chat_template(chat,add_generation_prompt=True,tokenize=False)
        # LLaVA-OneVision 4.45 processor's native video argument receives frames in RGB format.
        inputs=processor(text=prompt,videos=frames,return_tensors='pt')
        assert 'pixel_values_videos' in inputs,('Missing video tensor',list(inputs.keys()))
        inputs={k:(v.to('cuda:0') if hasattr(v,'to') else v) for k,v in inputs.items()}
        torch.cuda.reset_peak_memory_stats('cuda:0');torch.cuda.synchronize();start=time.perf_counter()
        with torch.inference_mode():output_ids=model.generate(**inputs,max_new_tokens=32,do_sample=False)
        torch.cuda.synchronize();elapsed=time.perf_counter()-start
        decoded=processor.decode(output_ids[0,inputs['input_ids'].shape[-1]:],skip_special_tokens=True)
        pred=choice(decoded);gold=str(rec['gold_choice'])
        peak=torch.cuda.max_memory_allocated('cuda:0')/1024**3
        peak_res=torch.cuda.max_memory_reserved('cuda:0')/1024**3
        row={'video_id':rec['video_id'],'qa_index':rec['qa_index'],'question':rec['question'],'task':rec['task'],
             'end_time_s':rec['end_time_s'],'gold_choice':gold,'prediction_text':decoded,'parsed_choice':pred,
             'correct':int(pred==gold),'sampled_frames':nframes,'duration_s':duration,
             'elapsed_s':elapsed,'peak_allocated_gb':peak,'peak_reserved_gb':peak_res,'status':'ok'}
        rows=[r for r in rows if (str(r['video_id']),int(r['qa_index']))!=key]
        rows.append(row)
        tmp=output.with_suffix('.tmp')
        with tmp.open('w',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows)
        tmp.replace(output)
        print('NATIVE_OK',processed,key,'frames',nframes,'correct',int(pred==gold),flush=True)
        del inputs,output_ids,frames
        torch.cuda.empty_cache()
    except Exception as exc:
        with output.with_suffix('.failures.txt').open('a') as f:
            f.write(f'{key}\\t{type(exc).__name__}: {exc}\\n')
        traceback.print_exc()
        print('NATIVE_FAILED',key,'aborting, results to date kept',flush=True)
        raise
print('NATIVE_FINISHED',len(old_rows()),'rows',flush=True)

"""
# Build independent 50-video worker records from the official annotation, NOT from any HERMES predicted answers.
ref=h50[4000]
path_by_vid={str(item['video_id']):item for item in matched50}
records=[]
for _,r in ref.iterrows():
    vid=str(r['video_id']); q=r['question']
    ann=path_by_vid[vid]
    qa_index=int(r['qa_index'])
    qa=ann['conversations'][qa_index]
    assert str(qa['question']).strip()==str(q).strip(),(vid,qa_index,'question mismatch')
    choices=qa.get('choices')
    # Match the exact answer options too; generic prompts alone are not IDs.
    csv_choices=ast.literal_eval(r['choices']) if isinstance(r['choices'],str) else r['choices']
    assert [str(t).strip() for t in csv_choices]==[str(t).strip() for t in choices],(
        vid,qa_index,'option mismatch')
    assert isinstance(choices,list) and len(choices)==4,(vid,q,'unexpected choices format')
    answers=[str(c).strip() for c in choices]; gold=str(qa['answer']).strip()
    hits=[i for i,s in enumerate(answers) if s==gold]
    assert len(hits)==1,(vid,q,'gold not uniquely found in choices')
    correct='ABCD'[hits[0]]
    if 'correct_choice' in r and str(r['correct_choice']).strip().upper()!=correct:
        raise AssertionError(f'Gold choice differs from HERMES result: {vid} {q}')
    t=qa.get('end_time')
    assert t is not None and float(t)>=0,(vid,q,'missing valid question time')
    records.append({'video_id':vid,'qa_index':qa_index,'question':q,'task':qa.get('task','Unknown'),
                    'video_path':ann['video_path'],'end_time_s':float(t),
                    'choices':choices,'gold_choice':correct})
assert len(records)==250 and len({(r['video_id'],r['qa_index']) for r in records})==250
NATIVE_DIR=RESULTS/'native_llava_50'; NATIVE_DIR.mkdir(parents=True,exist_ok=True)
NATIVE_RECORDS=DATA/'native_llava_50_worker_records.json'
NATIVE_RECORDS.write_text(json.dumps(records,indent=2))
NATIVE_CSV=NATIVE_DIR/'predictions.csv'
NATIVE_SCRIPT=EXPORTS/'native_llava_worker_py312.py'
NATIVE_SCRIPT.write_text(WORKER_SOURCE)
compile(NATIVE_SCRIPT.read_text(),str(NATIVE_SCRIPT),'exec')
print('Native records:',len(records),'Worker:',NATIVE_SCRIPT,'CSV:',NATIVE_CSV)


Native records: 250 Worker: /kaggle/working/HERMES_MASTER_EXPORTS/native_llava_worker_py312.py CSV: /kaggle/working/HERMES_MASTER/results/native_llava_50/predictions.csv


## 12. Native baseline smoke + resumable 250-question run

Smoke tests **one** question before continuing. If the locked model cannot process the video with full 0.5-FPS context, its failure is logged; the notebook continues to export valid HERMES analyses rather than reporting an invented baseline. **Do not interpret a missing or partial native control as completed research.**

In [30]:
from pathlib import Path

path = Path(NATIVE_SCRIPT)
text = path.read_text()

old = "inputs={k:(v.to('cuda:0') if hasattr(v,'to') else v) for k,v in inputs.items()}"

new = """inputs={
    k: (
        v.to(device='cuda:0', dtype=model.dtype)
        if torch.is_tensor(v) and v.is_floating_point()
        else v.to('cuda:0')
        if torch.is_tensor(v)
        else v
    )
    for k,v in inputs.items()
}"""

assert text.count(old) == 1, (
    "Expected input-transfer line not found. "
    "No changes were made."
)

path.write_text(text.replace(old, new))

# Keep the notebook's worker template consistent so it
# does not overwrite the fix if its creation cell is rerun.
if "WORKER_SOURCE" in globals():
    if WORKER_SOURCE.count(old) == 1:
        WORKER_SOURCE = WORKER_SOURCE.replace(old, new)

compile(path.read_text(), str(path), "exec")

print("PASS: Native LLaVA dtype patch applied.")
print("PASS: Worker syntax validated.")
print("Worker:", path)

PASS: Native LLaVA dtype patch applied.
PASS: Worker syntax validated.
Worker: /kaggle/working/HERMES_MASTER_EXPORTS/native_llava_worker_py312.py


In [31]:
NATIVE_LOG=LOGS/'native_llava_50.log'
NATIVE_SMOKE_LOG=LOGS/'native_smoke.log'
native_ok=False
if RUN_NATIVE_BASELINE_50:
    def native_cmd(limit):
        command=[PY312,NATIVE_SCRIPT,'--records',NATIVE_RECORDS,'--output',NATIVE_CSV,
                 '--model',MODEL_DIR,'--fps',str(FPS)]
        if limit:command+=['--limit',str(limit)]
        return [str(x) for x in command]
    # The first call writes one durable prediction. The second resumes it.
    for stage,lim,log in [('smoke',1,NATIVE_SMOKE_LOG),('full',0,NATIVE_LOG)]:
        print('Native',stage,'started: see',log)
        with log.open('w') as f:
            p=subprocess.run(native_cmd(lim),cwd=ROOT,env=SUBENV,stdout=f,stderr=subprocess.STDOUT)
        print('Native',stage,'return code',p.returncode)
        print(log.read_text(errors='replace')[-2200:])
        if p.returncode:
            (EXPORTS/'native_failure_status.txt').write_text(f'{stage}: failed. See {log}. Do not report a completed native baseline.\n')
            break
    else:native_ok=True
else:print('Native baseline disabled in configuration.')
if NATIVE_CSV.exists():
    native_df=pd.read_csv(NATIVE_CSV)
    if all(col in native_df for col in KEY) and len(native_df)==250 and not native_df.duplicated(KEY).any() and native_df.status.eq('ok').all():
        native_df['video_id']=native_df.video_id.astype(str)
        native_df['qa_index']=pd.to_numeric(native_df.qa_index).astype(int)
        hmeta=h50[4000].set_index(KEY).sort_index()
        nmeta=native_df.set_index(KEY).sort_index()
        assert hmeta.index.equals(nmeta.index),'Native and HERMES positional IDs differ'
        assert hmeta.question.astype(str).str.strip().equals(nmeta.question.astype(str).str.strip()),'Native questions differ'
        assert hmeta.correct_choice.astype(str).str.strip().str.upper().equals(nmeta.gold_choice.astype(str).str.strip().str.upper()),'Native gold answers differ'
        native_ok=True
    print('Native predicted rows:',len(native_df),'of 250; complete:',native_ok)
else:native_df=pd.DataFrame()


Native smoke started: see /kaggle/working/HERMES_MASTER_LOGS/native_smoke.log
Native smoke return code 0
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
NATIVE_OK 1 ('231', 0) frames 7 correct 0
NATIVE_FINISHED 1 rows

Native full started: see /kaggle/working/HERMES_MASTER_LOGS/native_llava_50.log
Native full return code 1
/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/temp/hermes_py312/lib/python3.12/site-packages/transformers/models/siglip/modeling_siglip.py", line 652, in forward
    hidden_states = self.mlp(hidden_states)
                    ^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/temp/hermes_py312/lib/python3.12/site-package

In [32]:
import json
import pandas as pd
from pathlib import Path

native = pd.read_csv(NATIVE_CSV)
records = json.loads(Path(NATIVE_RECORDS).read_text())

print("=== COMPLETED NATIVE RESULTS ===")
print("Saved predictions:", len(native))
print("Unique questions:",
      len(native.drop_duplicates(["video_id", "qa_index"])))

print("\nLast 8 successful predictions:")
cols = [
    "video_id", "qa_index", "sampled_frames",
    "peak_allocated_gb", "peak_reserved_gb"
]
print(native[cols].tail(8).to_string(index=False))

print("\nLargest successfully processed video prefixes:")
print(
    native[cols]
    .sort_values("sampled_frames", ascending=False)
    .head(8)
    .to_string(index=False)
)

completed = {
    (str(r.video_id), int(r.qa_index))
    for r in native.itertuples()
}

pending = [
    r for r in records
    if (str(r["video_id"]), int(r["qa_index"]))
    not in completed
]

print("\n=== FIRST UNFINISHED QUESTION ===")

if pending:
    r = pending[0]
    print("Video:", r["video_id"])
    print("Question index:", r["qa_index"])
    print("Question time:", r["end_time_s"], "seconds")
    print("Approximate frames at 0.5 FPS:",
          int(float(r["end_time_s"]) * 0.5) + 1)

failure_file = Path(NATIVE_CSV).with_suffix(".failures.txt")

if failure_file.exists():
    print("\n=== RECORDED FAILURE ===")
    print(failure_file.read_text(errors="replace")[-1200:])

=== COMPLETED NATIVE RESULTS ===
Saved predictions: 53
Unique questions: 53

Last 8 successful predictions:
 video_id  qa_index  sampled_frames  peak_allocated_gb  peak_reserved_gb
      289         0               5           2.162233          2.373047
      289         1              10           2.445776          2.748047
      289         2              34           3.805971          4.412109
      289         3              62           5.396744          6.398438
      289         4              99           7.487946          9.017578
      183         0               9           2.389125          2.673828
      183         1              72           5.961239          7.105469
      183         2             131           9.300478         11.251953

Largest successfully processed video prefixes:
 video_id  qa_index  sampled_frames  peak_allocated_gb  peak_reserved_gb
      118         4             158          10.836747         13.166016
      448         4             133      

In [33]:
import subprocess
import json
import pandas as pd
from pathlib import Path

# Verify the next unfinished question.
saved = pd.read_csv(NATIVE_CSV)
records = json.loads(Path(NATIVE_RECORDS).read_text())

done = {
    (str(r.video_id), int(r.qa_index))
    for r in saved.itertuples()
    if r.status == "ok"
}

pending = next(
    r for r in records
    if (str(r["video_id"]), int(r["qa_index"])) not in done
)

key = (str(pending["video_id"]), int(pending["qa_index"]))
print("Saved predictions:", len(saved))
print("Next question:", key)

assert key == ("183", 3), (
    "The next pending question changed; inspect before retrying."
)

# Change only the GPU memory allocator.
retry_env = dict(SUBENV)
retry_env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

command = [
    str(PY312),
    str(NATIVE_SCRIPT),
    "--records", str(NATIVE_RECORDS),
    "--output", str(NATIVE_CSV),
    "--model", str(MODEL_DIR),
    "--fps", str(FPS),
    "--limit", "1",
]

retry_log = LOGS / "native_199frame_retry.log"

print("\nRetrying exactly one question...")

with retry_log.open("w") as f:
    result = subprocess.run(
        command,
        cwd=ROOT,
        env=retry_env,
        stdout=f,
        stderr=subprocess.STDOUT
    )

print("Return code:", result.returncode)
print(retry_log.read_text(errors="replace")[-2500:])

updated = pd.read_csv(NATIVE_CSV)
print("\nSaved native predictions:", len(updated), "/ 250")

Saved predictions: 53
Next question: ('183', 3)

Retrying exactly one question...
Return code: 0
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Token indices sequence length is longer than the specified maximum sequence length for this model (39068 > 32768). Running this sequence through the model will result in indexing errors
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
NATIVE_OK 1 ('183', 3) frames 199 correct 0
NATIVE_FINISHED 54 rows


Saved native predictions: 54 / 250


In [34]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

records = pd.DataFrame(
    json.loads(Path(NATIVE_RECORDS).read_text())
)

saved = pd.read_csv(NATIVE_CSV)

config = json.loads(
    (Path(MODEL_DIR) / "config.json").read_text()
)

text_config = config.get("text_config", {})
context_limit = text_config.get(
    "max_position_embeddings",
    32768
)

# Estimate frame counts from timestamps.
# Actual counts can be smaller when timestamps exceed video duration.
records["estimated_frames"] = (
    np.floor(
        pd.to_numeric(records["end_time_s"]) * float(FPS)
    ).astype(int) + 1
)

# Observed from the successful 199-frame question.
observed_tokens = 39068
observed_frames = 199
tokens_per_frame_approx = observed_tokens / observed_frames

records["estimated_tokens"] = (
    records["estimated_frames"] * tokens_per_frame_approx
).round().astype(int)

records["potential_context_overflow"] = (
    records["estimated_tokens"] > context_limit
)

print("=== NATIVE BASELINE PREFLIGHT ===")
print("Saved predictions:", len(saved), "/ 250")
print("Model context limit:", context_limit)
print("Observed tokens at 199 frames:", observed_tokens)
print("Approximate tokens/frame:", round(tokens_per_frame_approx, 1))

print("\nTotal questions:", len(records))
print(
    "Potentially over context limit:",
    records["potential_context_overflow"].sum()
)

print("\nLongest estimated prefixes:")
print(
    records[
        ["video_id", "qa_index", "estimated_frames",
         "estimated_tokens", "potential_context_overflow"]
    ]
    .sort_values("estimated_frames", ascending=False)
    .head(15)
    .to_string(index=False)
)

=== NATIVE BASELINE PREFLIGHT ===
Saved predictions: 54 / 250
Model context limit: 32768
Observed tokens at 199 frames: 39068
Approximate tokens/frame: 196.3

Total questions: 250
Potentially over context limit: 76

Longest estimated prefixes:
video_id  qa_index  estimated_frames  estimated_tokens  potential_context_overflow
     222         4               346             67927                        True
     402         4               335             65768                        True
     492         4               324             63608                        True
     256         4               322             63216                        True
     139         4               317             62234                        True
      80         4               309             60663                        True
     216         4               309             60663                        True
      93         4               305             59878                        True
     171 

In [35]:
from pathlib import Path
import json
import pandas as pd

native = pd.read_csv(NATIVE_CSV)

status = {
    "experiment": "native_llava_full_prefix_50",
    "status": "incomplete_context_and_memory_limitations",
    "completed_questions": len(native),
    "expected_questions": 250,
    "declared_context_limit": 32768,
    "observed_overlength_tokens": 39068,
    "observed_overlength_frames": 199,
    "estimated_potential_overflows": 76,
    "notes": (
        "Partial native LLaVA results only. "
        "No full-baseline accuracy should be reported. "
        "The 76 potential overflows are estimates, not verified counts."
    )
}

EXPORTS.mkdir(parents=True, exist_ok=True)

output = EXPORTS / "native_baseline_feasibility.json"
output.write_text(json.dumps(status, indent=2))

print("Saved:", output)
print("Native predictions preserved:", len(native))
print("Status: INCOMPLETE — not a valid full baseline")

Saved: /kaggle/working/HERMES_MASTER_EXPORTS/native_baseline_feasibility.json
Native predictions preserved: 54
Status: INCOMPLETE — not a valid full baseline


In [36]:
from pathlib import Path
import subprocess

WORK = Path("/kaggle/working")

print("=== KAGGLE OUTPUT INVENTORY ===")

subprocess.run(
    "du -sh /kaggle/working/* 2>/dev/null | sort -h",
    shell=True
)

print("\n=== RESEARCH CSV FILES ===")

for p in sorted(WORK.rglob("*.csv")):
    if ".git" not in p.parts:
        print(
            f"{p.stat().st_size / 1024:.1f} KB |",
            p.relative_to(WORK)
        )

print("\n=== EXISTING ZIP ARCHIVES ===")

for p in sorted(WORK.glob("*.zip")):
    print(
        f"{p.stat().st_size / 1024**2:.2f} MB |",
        p.name
    )

=== KAGGLE OUTPUT INVENTORY ===
84K	/kaggle/working/HERMES_KV4000_PRESERVED.zip
360K	/kaggle/working/HERMES_MASTER_EXPORTS
2.4M	/kaggle/working/HERMES_MASTER_LOGS
17M	/kaggle/working/HERMES_MASTER

=== RESEARCH CSV FILES ===
9.6 KB | HERMES_MASTER/results/native_llava_50/predictions.csv
90.4 KB | HERMES_MASTER/results/streamingbench_50_kv4000/1_0.csv
90.4 KB | HERMES_MASTER/results/streamingbench_50_kv50000/1_0.csv
90.5 KB | HERMES_MASTER/results/streamingbench_50_kv6000/1_0.csv
16.3 KB | HERMES_MASTER_EXPORTS/all_three_50_correctness.csv
0.1 KB | HERMES_MASTER_EXPORTS/hermes_50_accuracy.csv
292.3 KB | HERMES_MASTER_EXPORTS/paired_50_casebook.csv
0.5 KB | HERMES_MASTER_EXPORTS/paired_50_tests.csv
0.1 KB | HERMES_MASTER_EXPORTS/recovery_manifest.csv

=== EXISTING ZIP ARCHIVES ===
0.08 MB | HERMES_KV4000_PRESERVED.zip


In [37]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
import os, json, hashlib
import pandas as pd

WORK = Path("/kaggle/working")
ROOT = WORK / "HERMES_MASTER"

ZIP = WORK / (
    "HERMES_RESEARCH_FINAL_"
    + datetime.now().strftime("%Y%m%d_%H%M")
    + ".zip"
)

# Confirm the three core benchmark files exist and are complete.
required = [
    ROOT / "results" / f"streamingbench_50_kv{kv}" / "1_0.csv"
    for kv in (4000, 6000, 50000)
]

for p in required:
    assert p.exists(), f"Missing research result: {p}"
    rows = len(pd.read_csv(p))
    assert rows == 250, f"{p.name}: expected 250, found {rows}"
    print("Verified:", p.parent.name, rows, "rows")

# Source and research output directories.
folders = [
    ROOT,
    WORK / "HERMES_MASTER_EXPORTS",
    WORK / "HERMES_MASTER_LOGS",
]

excluded_dirs = {
    ".git", "__pycache__", ".cache", "models",
    "hf_cache", "model_cache", "videos",
    "frames", "download_cache"
}

excluded_ext = {
    ".pyc", ".mp4", ".avi", ".mkv", ".mov",
    ".webm", ".safetensors", ".onnx",
    ".pt", ".pth", ".bin", ".zip"
}

manifest = {
    "created": datetime.now().isoformat(),
    "project": "HERMES streaming video understanding reproduction",
    "native_baseline_status": "incomplete",
    "notes": [
        "KV=50000 is a HERMES large-budget control, not native LLaVA.",
        "Native baseline predictions are partial.",
        "Preserve original raw results for reproducibility."
    ],
    "files": {}
}

selected = {}

for folder in folders:
    if not folder.is_dir():
        continue

    for parent, dirs, filenames in os.walk(folder):
        dirs[:] = [
            d for d in dirs
            if d not in excluded_dirs
            and not d.startswith(".")
        ]

        for filename in filenames:
            p = Path(parent) / filename

            if not p.is_file() or p.is_symlink():
                continue

            if p.suffix.lower() in excluded_ext:
                continue

            if p.stat().st_size > 50 * 1024**2:
                print("Skipped large file:", p)
                continue

            relative = p.relative_to(WORK).as_posix()
            selected[relative] = p

# Include notebook source if Kaggle has saved one in working.
for p in WORK.glob("*.ipynb"):
    selected[p.relative_to(WORK).as_posix()] = p

# Preserve additional referenced files if stored outside the three folders.
for name in ("NATIVE_RECORDS", "NATIVE_SCRIPT", "NATIVE_CSV"):
    value = globals().get(name)
    if value is None:
        continue
    p = Path(value)
    if not p.is_file():
        continue
    if p.is_relative_to(WORK):
        arcname = p.relative_to(WORK).as_posix()
    else:
        arcname = "supplemental/" + name.lower() + p.suffix
    selected[arcname] = p

with ZipFile(ZIP, "w", ZIP_DEFLATED, compresslevel=6) as archive:
    for relative, p in sorted(selected.items()):
        archive.write(p, relative)

        h = hashlib.sha256()
        with p.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)

        manifest["files"][relative] = {
            "bytes": p.stat().st_size,
            "sha256": h.hexdigest()
        }

    archive.writestr(
        "RESEARCH_MANIFEST.json",
        json.dumps(manifest, indent=2)
    )

    archive.writestr(
        "README_FIRST.txt",
        """HERMES REPRODUCTION RESEARCH BACKUP

Completed:
- 50-video HERMES KV=4000 evaluation
- 50-video HERMES KV=6000 evaluation
- 50-video HERMES KV=50000 evaluation
- Paired statistical analysis
- Raw prediction CSVs and logs

Native LLaVA:
- Partial evaluation only (54 predictions reported)
- Do not treat as a completed 250-question baseline

KV=50000:
- Large-budget HERMES control
- NOT the original LLaVA baseline

See RESEARCH_MANIFEST.json for individual file hashes.
"""
    )

# Validate archive integrity and each archived file's checksum.
with ZipFile(ZIP) as archive:
    assert archive.testzip() is None, "ZIP integrity failed"

    for relative, metadata in manifest["files"].items():
        actual_hash = hashlib.sha256(
            archive.read(relative)
        ).hexdigest()

        assert actual_hash == metadata["sha256"], (
            f"Checksum mismatch: {relative}"
        )

print("\n" + "=" * 55)
print("FINAL RESEARCH BACKUP CREATED")
print("=" * 55)
print("File:", ZIP)
print("Size:", round(ZIP.stat().st_size / 1024**2, 2), "MB")
print("Research files:", len(selected))
print("ZIP integrity: PASS")
print("SHA-256 verification: PASS")

native_csv = ROOT / "results/native_llava_50/predictions.csv"
if native_csv.exists():
    print("Native predictions:", len(pd.read_csv(native_csv)), "/ 250")

print("\nOriginal research files remain untouched.")

Verified: streamingbench_50_kv4000 250 rows
Verified: streamingbench_50_kv6000 250 rows
Verified: streamingbench_50_kv50000 250 rows

FINAL RESEARCH BACKUP CREATED
File: /kaggle/working/HERMES_RESEARCH_FINAL_20261016_2325.zip
Size: 2.41 MB
Research files: 47
ZIP integrity: PASS
SHA-256 verification: PASS
Native predictions: 54 / 250

Original research files remain untouched.


In [38]:
from pathlib import Path

old = Path("/kaggle/working/HERMES_KV4000_PRESERVED.zip")

assert ZIP.exists()

with ZipFile(ZIP) as z:
    assert z.testzip() is None

if old.exists():
    old.unlink()
    print("Removed redundant old ZIP:", old.name)

print("Final research ZIP retained:", ZIP.name)

Removed redundant old ZIP: HERMES_KV4000_PRESERVED.zip
Final research ZIP retained: HERMES_RESEARCH_FINAL_20261016_2325.zip


## 13. Native vs HERMES comparisons (only if all 250 native predictions are complete)

Native full-prefix measurements must not be reported as equivalent streaming latency; report its `elapsed_s` as *native generate time*, not HERMES TTFT. Statistical comparisons share the question set but protocol deviations should be disclosed.

In [ ]:
if native_ok:
    n=native_df.copy();n['video_id']=n.video_id.astype(str);n['question']=n.question.astype(str).str.strip()
    n['correct']=pd.to_numeric(n.correct).astype(int)
    assert len(n)==250 and not n.duplicated(KEY).any()
    reports=[];case_frames=[]
    for kv in KV_50:
        x,c=paired(h50[kv],n,f'HERMES_{kv}','native_LLaVA_fullprefix')
        reports.append(x);case_frames.append(c)
    baseline_comparisons=pd.DataFrame(reports)
    baseline_comparisons.to_csv(EXPORTS/'native_vs_hermes_paired.csv',index=False)
    pd.concat(case_frames,ignore_index=True).to_csv(EXPORTS/'native_vs_hermes_casebook.csv',index=False)
    display(baseline_comparisons.round(4))
    print('Native mean accuracy:',round(100*n.correct.mean(),2),'%')
else:print('Native comparison is NOT available: native full-prefix baseline incomplete/failed. No placeholder scores will be written.')


## 14. Optional full StreamingBench evaluation (~498 videos, ~2490 QA)

`RUN_FULL_HERMES=False` by default because 10 multi-GB archives and two HERMES budgets can exceed one Kaggle GPU session. When enabled, the notebook processes up to `MAX_NEW_SHARDS_THIS_SESSION` *new* shards, verifying complete result CSVs and deleting temporary extracted videos after both KV budgets. **Save outputs and reattach them as Kaggle input before later sessions.** If a native baseline fails or is infeasible, you can still study HERMES 4K and 6K on the full benchmark, but must report the missing baseline.

In [ ]:
FULL_ARCHIVES=[f'Real-Time Visual Understanding_{a}-{b}.zip' for a,b in [(1,50),(51,100),(101,150),(151,200),(201,250),(251,300),(301,350),(351,400),(401,450),(451,500)]]
full_run_rows=[]
if RUN_FULL_HERMES:
    new_shards=0
    for name in FULL_ARCHIVES:
        tag=re.search(r'(\d+-\d+)\.zip$',name).group(1)
        expected_items=[x for x in annotations if (lambda m: m and int(tag.split('-')[0])<=int(m.group(1))<=int(tag.split('-')[1]))(re.search(r'sample_(\d+)_real\.mp4',x['video_path']))]
        expected_n=len(expected_items)
        expected_q=sum(len(x['conversations']) for x in expected_items)
        if not expected_n:
            print('No matching annotations for shard',tag);continue
        if all((RESULTS/f'full_{tag}_kv{kv}'/'1_0.csv').exists() for kv in FULL_BUDGETS):
            try:
                for kv in FULL_BUDGETS:read_hermes_csv(RESULTS/f'full_{tag}_kv{kv}'/'1_0.csv',expected_n,expected_q)
                print('Already completed full shard',tag);continue
            except Exception as e:raise RuntimeError(f'Unusable existing output for {tag}; fix before resuming') from e
        if new_shards>=MAX_NEW_SHARDS_THIS_SESSION:
            print('Session shard limit reached. Save Kaggle outputs and resume next session.');break
        extracted=fetch_extract(name)
        matched=annotation_for_directory(extracted)
        # Validate against exact numeric file/sample IDs to avoid including wrong videos.
        assert len(matched)==expected_n,(tag,len(matched),expected_n)
        assert sum(len(x['conversations']) for x in matched)==expected_q
        path=DATA/f'full_{tag}.json';path.write_text(json.dumps(matched,indent=2))
        for kv in FULL_BUDGETS:
            d=run_hermes(path,f'full_{tag}_kv{kv}',kv,expected_n,expected_q)
            full_run_rows.append({'shard':tag,'kv':kv,'video_count':len(matched),
                                  'question_count':len(d),'accuracy_pct':100*d.correct.mean()})
        new_shards+=1
        if not RETAIN_EXTRACTED_SHARDS:
            shutil.rmtree(extracted,ignore_errors=True)
            print('Removed temporary extracted videos for',tag)
else:print('Full benchmark is available in this notebook but disabled; set RUN_FULL_HERMES=True when ready.')
if full_run_rows:
    pd.DataFrame(full_run_rows).to_csv(EXPORTS/'full_new_shard_runs.csv',index=False)


## 15. Full-benchmark statistics from available **validated** shard CSVs

Partial coverage is reported as partial coverage. Only identical question keys are compared in paired full-benchmark analysis.

In [ ]:
full_data={}
for kv in FULL_BUDGETS:
    frames=[]
    for tag in [s.rsplit('_',1)[-1].replace('.zip','') for s in FULL_ARCHIVES]:
        p=RESULTS/f'full_{tag}_kv{kv}'/'1_0.csv'
        if p.exists():
            d=read_hermes_csv(p)
            frames.append(d)
    if frames:
        full_data[kv]=pd.concat(frames,ignore_index=True)
        assert not full_data[kv].duplicated(KEY).any(),f'Duplicate full keys KV={kv}'
    else:full_data[kv]=pd.DataFrame()
summary=[]
for kv,d in full_data.items():
    if len(d):
        summary.append({'method':f'HERMES KV={kv}','videos_observed':d.video_id.nunique(),
                        'questions_observed':len(d),'accuracy_pct':100*d.correct.mean(),
                        'complete_498_annotation_videos':d.video_id.nunique()==len(annotations)})
if summary:
    full_summary=pd.DataFrame(summary);display(full_summary)
    full_summary.to_csv(EXPORTS/'full_benchmark_accuracy_partial_or_complete.csv',index=False)
else:
    full_summary=pd.DataFrame();print('No full-benchmark shard results yet.')
if all(len(full_data[kv]) for kv in FULL_BUDGETS):
    da,db=(full_data[kv] for kv in FULL_BUDGETS)
    ka=set(map(tuple,da[KEY].values));kb=set(map(tuple,db[KEY].values))
    if ka==kb:
        fr,fc=paired(da,db,'HERMES_full_4K','HERMES_full_6K')
        pd.DataFrame([fr]).to_csv(EXPORTS/'full_paired_4k_vs_6k.csv',index=False)
        fc.to_csv(EXPORTS/'full_pair_casebook.csv',index=False)
        display(pd.DataFrame([fr]).round(4))
    else:print('Full 4K/6K coverage differs; not reporting misleading paired analysis. A:',len(ka),'B:',len(kb))


## 16. Task summaries, documented memory/latency metrics and report figures

In [ ]:
metrics=[]
for kv,df in h50.items():
    meta=df.groupby('task')['correct'].agg(['sum','count']).reset_index() if 'task' in df else pd.DataFrame()
    if len(meta):
        meta['accuracy_pct']=100*meta['sum']/meta['count'];meta['method']=f'HERMES KV={kv}'
        metrics.append(meta)
if metrics:
    pd.concat(metrics).to_csv(EXPORTS/'task_accuracy_50.csv',index=False)

# Preserve logs from the old /kaggle/working/HERMES if available; do not fabricate efficiency metrics.
for kv in KV_50:
    log=LOGS/f'streamingbench_50_kv{kv}.log'
    if not log.exists():
        candidates=[WORK/'HERMES'/f'streamingbench_50_kv{kv}.log',WORK/'HERMES_FINAL'/f'streamingbench_50_kv{kv}.log']
        for src in candidates:
            if src.exists():shutil.copy2(src,log);break
log_rows=[]
for kv in KV_50:
    log=LOGS/f'streamingbench_50_kv{kv}.log'
    if not log.exists():
        log_rows.append({'method':f'HERMES KV={kv}','log_available':False})
        continue
    text=log.read_text(errors='replace')
    log_rows.append({'method':f'HERMES KV={kv}','log_available':True,
         'compression_mentions':len(re.findall(r'Applying KV-Cache compression|Layer-wise shrink completed',text)),
         'reported_TTFT_tokens':len(re.findall('TTFT',text))})
pd.DataFrame(log_rows).to_csv(EXPORTS/'log_availability_and_event_counts.csv',index=False)

labels=[f'KV={k}' for k in KV_50]; values=[100*h50[k].correct.mean() for k in KV_50]
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(labels,values)
ax.set_ylabel('Accuracy (%)');ax.set_title('HERMES: 50 videos / 250 paired questions')
for idx,v in enumerate(values):ax.text(idx,v+.7,f'{v:.1f}%',ha='center')
ax.set_ylim(0,max(values)+12);fig.tight_layout()
fig.savefig(FIGURES/'hermes_50_accuracy.png',dpi=160);plt.show();plt.close(fig)
print('Metrics/figures written under',EXPORTS)


## 17. Provenance manifest, reproducibility ZIP, integrity test

Exports raw available results, logs, analysis, question annotations, model-worker source and a manifest with source SHA, package revision and input hashes. **Video files, virtual environments and model weights are excluded** to avoid a massive archive. A passed ZIP check verifies archive integrity, *not* that all research experiments completed.

In [ ]:
MANIFEST={'generated_at_utc':datetime.now(timezone.utc).isoformat(),
  'python_kernel':sys.version.split()[0],'native_python':run([PY312,'--version']).stdout.strip(),
  'hermes_git_sha':run(['git','rev-parse','HEAD'],cwd=ROOT).stdout.strip(),
  'hermes_tracked_diff':run(['git','diff','--stat','HEAD'],cwd=ROOT).stdout.strip(),
  'transformers_git_sha_requested':TF_SHA,'model_id':MODEL_ID,'model_snapshot_id':MODEL_SNAPSHOT.name,
  'sample_fps':FPS,'seed':SEED,'run_native_requested':RUN_NATIVE_BASELINE_50,
  'native_baseline_complete':bool(native_ok),'full_requested':RUN_FULL_HERMES,
  'full_videos_observed_by_budget':{str(k):int(v.video_id.nunique()) if len(v) else 0 for k,v in full_data.items()},
  'source_50_csvs_complete':all(valid50(wanted_path(k)) for k in KV_50),
  'hermes_csv_sha256':{str(k):sha256(wanted_path(k)) for k in KV_50},
  'annotation_50_sha256':sha256(ANNOTATION_50),
  'native_worker_sha256':sha256(NATIVE_SCRIPT),
  'protocol_caveat':'Native full-prefix LLaVA and online HERMES are not identical inference protocols.'}
(EXPORTS/'experiment_manifest.json').write_text(json.dumps(MANIFEST,indent=2))
# Identify all small data and logs to include without copying model weights or video assets.
zip_path=WORK/'HERMES_MASTER_RESEARCH_RESULTS.zip'
files_to_zip=[]
for folder in [EXPORTS,RESULTS,LOGS]:
    for p in folder.rglob('*'):
        if not p.is_file() or p.is_symlink():continue
        if p.suffix.lower() in {'.mp4','.mkv','.avi','.mov','.safetensors','.bin','.zip'}:continue
        if p.stat().st_size>100*1024*1024:continue
        files_to_zip.append(p)
for p in [ANNOTATION_50,ANNOTATION_10,NATIVE_RECORDS,ROOT/'data'/'streamingbench'/'streamingbench_realtime.json']:
    if p.exists() and p not in files_to_zip:files_to_zip.append(p)
files_to_zip=sorted(set(files_to_zip))
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in files_to_zip:
        # Preserve results and analysis in separate namespaces.
        arc=(Path('experiment_outputs')/p.relative_to(ROOT)) if ROOT in p.parents else (Path('report')/p.relative_to(EXPORTS) if EXPORTS in p.parents else Path('external')/p.name)
        z.write(p,str(arc))
with zipfile.ZipFile(zip_path) as z:
    bad=z.testzip();entries=z.namelist()
assert bad is None,('Archive CRC failed:',bad)
(EXPORTS/'final_archive_sha256.txt').write_text(f'{sha256(zip_path)}  {zip_path.name}\n')
print('RESEARCH ZIP:',zip_path,'size MiB:',round(zip_path.stat().st_size/1024**2,2),'files:',len(entries),'CRC: PASS')
print('Native full-prefix completed:',native_ok,'Full dataset coverage:',{k:len(v) for k,v in full_data.items()})


## 18. What constitutes completion and how to resume

**The notebook runs from first to last cell**, but externally sourced multi-GB assets, Kaggle session time limits, or OOM constraints can prevent all experiments completing in one session. Research outputs are always based on actual CSVs, never placeholder scores.

- **50-video paired analysis completed:** `paired_50_tests.csv`, `paired_50_casebook.csv`, `all_three_50_correctness.csv` exist and each underlying 50-video CSV has exactly 250 uniquely keyed questions.
- **Native LLaVA completed:** `results/native_llava_50/predictions.csv` has 250 rows, correct keys and a successful native smoke test; if unavailable, native comparisons remain absent. The native full-prefix protocol must not be conflated with an equivalent online-streaming baseline.
- **Full 498 videos:** requires all official real-time archives, 4K and 6K results for every annotation record, and repeated Kaggle runs if necessary. Turn on `RUN_FULL_HERMES=True` and adjust `MAX_NEW_SHARDS_THIS_SESSION` only after confirming GPU time and disk capacity.
- **Persistence:** use **Save Version → Save & Run All / Output** and download the research ZIP. Reattach prior output directories or the ZIP in **Add Input** when resuming. The model and video files intentionally remain temporary, so are redownloaded if the environment resets.
- **Statistical language:** a nonsignificant paired test does not establish equivalence or non-inferiority. Comparison of CPU tables and efficiency logs should disclose any prompt, frame, time cutoff and evaluation differences.